<a href="https://colab.research.google.com/github/habibiputrar/DeepLearning_B_2411531001_Habibi-Putra-Rizqullah/blob/main/Praktikum6/2411531001_Habibi_Putra_Rizqullah_Program6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum Deep Learning 6
## Transformer dan Attention Mechanism untuk Pemrosesan Bahasa

**Nama:** Habibi Putra Rizqullah  
**NIM:** 2411531001  
**Kelas:** Deep Learning B


# BAGIAN I PRAKTIKUM

## 1. Persiapan Library

Library yang digunakan adalah Transformers dan PyTorch.

In [ ]:
%pip install -q transformers torch

## 2. Import Library

`BertTokenizer` memecah kalimat menjadi token. `BertModel` digunakan untuk memperoleh embedding.

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

## 3. Memuat Tokenizer dan Model

Model yang digunakan adalah `bert-base-uncased`. BERT ini sudah dilatih sebelumnya, jadi tidak perlu dilatih dari awal.

In [ ]:
# 1. Load tokenizer dan model pretrained
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 4. Tokenisasi Kalimat

Kalimat dipecah menjadi token. Setiap token kemudian diubah menjadi ID sebagai input BERT.

In [ ]:
# 2. Tokenisasi kalimat
text = "Deep learning with Transformers is powerful."
inputs = tokenizer(text, return_tensors="pt")

## 5. Forward Pass

Input dimasukkan ke BERT untuk mendapatkan embedding setiap token.

In [ ]:
# 3. Forward pass ke model
outputs = model(**inputs)

## 6. Ekstraksi Embedding

`last_hidden_state` berisi embedding dari lapisan terakhir BERT. Bentuk tensornya menunjukkan jumlah kalimat, jumlah token, dan ukuran embedding.

In [ ]:
# 4. Ekstraksi embedding
last_hidden_states = outputs.last_hidden_state

print("Shape embedding:", last_hidden_states.shape)

Shape embedding: torch.Size([1, 9, 768])


### Hasil Dimensi Embedding

Output `torch.Size([1, 9, 768])` menunjukkan bahwa ada 1 kalimat dengan 9 token. Setiap token diwakili oleh vektor berisi 768 angka. Jumlah 9 token ini sudah termasuk `[CLS]` dan `[SEP]`.

## 7. Menampilkan Token dan Vektor

Lima nilai pertama dari setiap vektor ditampilkan agar output lebih mudah dibaca.

In [ ]:
# 5. Visualisasi sederhana token dan vektornya
tokens = tokenizer.tokenize(text)
print("\nToken dan panjang embedding:")
for i, token in enumerate(tokens):
    print(f"{token}: {last_hidden_states[0][i][:5]}")  # hanya tampilkan 5 nilai pertama


Token dan panjang embedding:
deep: tensor([-0.1354, -0.0223, -0.0507,  0.1804, -0.4551], grad_fn=<SliceBackward0>)
learning: tensor([-0.2287, -0.4056,  0.3421,  0.4721,  0.1263], grad_fn=<SliceBackward0>)
with: tensor([-0.0771,  0.2034,  0.2739,  0.4392,  0.0373], grad_fn=<SliceBackward0>)
transformers: tensor([-0.0292,  0.5713,  0.0177,  0.4645,  0.0245], grad_fn=<SliceBackward0>)
is: tensor([ 1.7758, -0.0033, -0.4201,  0.0374,  0.5116], grad_fn=<SliceBackward0>)
powerful: tensor([-0.0763, -0.1152,  0.5550,  0.3940,  0.2432], grad_fn=<SliceBackward0>)
.: tensor([ 0.2802, -0.3462,  0.5860,  0.3429,  0.2341], grad_fn=<SliceBackward0>)


### Hasil Tampilan Token

Ada 7 token yang ditampilkan, yaitu `deep`, `learning`, `with`, `transformers`, `is`, `powerful`, dan tanda titik. Namun, vektor pada baris `deep` masih milik `[CLS]`, karena token khusus itu berada di posisi pertama input model. Akibatnya, nama token dan vektornya bergeser satu posisi.

## 8. Memeriksa Posisi Token

Token diambil langsung dari `input_ids` agar urutannya sama dengan embedding.

In [ ]:
tokens_input = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())
print("Token tanpa token khusus:", tokens)
print("Token input model:", tokens_input)
print("Jumlah token tanpa token khusus:", len(tokens))
print("Jumlah token input model:", len(tokens_input))
print("Batch size:", last_hidden_states.shape[0])
print("Panjang token:", last_hidden_states.shape[1])
print("Dimensi embedding:", last_hidden_states.shape[2])

print("\nToken dan lima nilai pertama embedding yang sesuai:")
for i, token in enumerate(tokens_input):
    print(f"{token}: {last_hidden_states[0, i, :5].detach()}")

Token tanpa token khusus: ['deep', 'learning', 'with', 'transformers', 'is', 'powerful', '.']
Token input model: ['[CLS]', 'deep', 'learning', 'with', 'transformers', 'is', 'powerful', '.', '[SEP]']
Jumlah token tanpa token khusus: 7
Jumlah token input model: 9
Batch size: 1
Panjang token: 9
Dimensi embedding: 768

Token dan lima nilai pertama embedding yang sesuai:
[CLS]: tensor([-0.1354, -0.0223, -0.0507,  0.1804, -0.4551])
deep: tensor([-0.2287, -0.4056,  0.3421,  0.4721,  0.1263])
learning: tensor([-0.0771,  0.2034,  0.2739,  0.4392,  0.0373])
with: tensor([-0.0292,  0.5713,  0.0177,  0.4645,  0.0245])
transformers: tensor([ 1.7758, -0.0033, -0.4201,  0.0374,  0.5116])
is: tensor([-0.0763, -0.1152,  0.5550,  0.3940,  0.2432])
powerful: tensor([ 0.2802, -0.3462,  0.5860,  0.3429,  0.2341])
.: tensor([ 0.4932,  0.1111, -0.3156,  0.1875, -0.0699])
[SEP]: tensor([ 0.9405,  0.2103, -0.0866,  0.4300, -0.2973])


### Hasil Pemeriksaan Token

Kalimat menghasilkan 7 token biasa dan 9 token setelah ditambah `[CLS]` serta `[SEP]`. Tanda titik ikut dihitung sebagai satu token. Hasil ini sesuai dengan bentuk embedding `[1, 9, 768]`.

Setelah urutannya disesuaikan, lima nilai awal vektor `deep` adalah `[-0.2287, -0.4056, 0.3421, 0.4721, 0.1263]`. Nilainya berbeda dari baris `deep` pada tampilan sebelumnya karena posisi tokennya sudah diperbaiki. Angka negatif tidak berarti kata tersebut memiliki sentimen negatif; angka itu merupakan bagian dari representasi token.

# BAGIAN II TUGAS INDIVIDU

Kemiripan dua kalimat dihitung menggunakan cosine similarity. Vektor kalimat diperoleh dari rata-rata embedding token, tanpa padding, `[CLS]`, dan `[SEP]`.

## 1. Pasangan Kalimat

Ada tiga pasangan kalimat: topik serupa, topik berbeda, serta topik sama dengan sentimen berlawanan.

In [ ]:
pasangan_kalimat = [
    ("I love AI.", "AI is great.", "Kedua kalimat menyatakan pandangan positif tentang AI."),
    ("The sky is blue.", "I love pizza.", "Kedua kalimat membahas topik yang berbeda."),
    ("I love this movie.", "I hate this movie.", "Topiknya sama, tetapi sentimennya berlawanan."),
]

for nomor, (kalimat_1, kalimat_2, _) in enumerate(pasangan_kalimat, 1):
    print(f"Pasangan {nomor}")
    print("Kalimat 1:", kalimat_1)
    print("Kalimat 2:", kalimat_2, "\n")

Pasangan 1
Kalimat 1: I love AI.
Kalimat 2: AI is great. 

Pasangan 2
Kalimat 1: The sky is blue.
Kalimat 2: I love pizza. 

Pasangan 3
Kalimat 1: I love this movie.
Kalimat 2: I hate this movie. 



## 2. Embedding Kalimat

Embedding setiap token dirata-ratakan menjadi satu vektor kalimat. `model.eval()` menonaktifkan dropout agar perhitungan lebih konsisten.

In [ ]:
import torch.nn.functional as F

model.eval()

def embedding_kalimat(daftar_kalimat):
    encoded = tokenizer(
        daftar_kalimat,
        padding=True,
        truncation=True,
        max_length=model.config.max_position_embeddings,
        return_special_tokens_mask=True,
        return_tensors="pt"
    )
    special_tokens_mask = encoded.pop("special_tokens_mask")
    with torch.no_grad():
        hidden = model(**encoded).last_hidden_state

    mask_isi = (encoded["attention_mask"] * (1 - special_tokens_mask))
    mask_isi = mask_isi.unsqueeze(-1).to(hidden.dtype)
    jumlah_token = mask_isi.sum(dim=1).clamp(min=1)
    return (hidden * mask_isi).sum(dim=1) / jumlah_token

## 3. Cosine Similarity

Cosine similarity dihitung dari dua vektor kalimat. Semakin mendekati 1, semakin searah kedua vektor tersebut.

In [ ]:
hasil_perbandingan = []
for kalimat_1, kalimat_2, interpretasi in pasangan_kalimat:
    embedding = embedding_kalimat([kalimat_1, kalimat_2])
    similarity = F.cosine_similarity(
        embedding[0].unsqueeze(0), embedding[1].unsqueeze(0), dim=1
    ).item()
    hasil_perbandingan.append({
        "Kalimat 1": kalimat_1,
        "Kalimat 2": kalimat_2,
        "Cosine Similarity": similarity,
        "Interpretasi": interpretasi,
    })
    print(f"{kalimat_1} | {kalimat_2} | {similarity:.4f}")

I love AI. | AI is great. | 0.7157
The sky is blue. | I love pizza. | 0.6160
I love this movie. | I hate this movie. | 0.9002


### Hasil Cosine Similarity

Pasangan “I love AI.” dan “AI is great.” memperoleh skor **0,7157**. Keduanya membahas AI dengan pandangan positif. Pasangan “The sky is blue.” dan “I love pizza.” mendapat skor **0,6160**, lebih rendah karena topiknya berbeda.

Skor tertinggi justru muncul pada “I love this movie.” dan “I hate this movie.”, yaitu **0,9002**. Kedua kalimat memiliki susunan dan sebagian besar kata yang sama, tetapi kata `love` dan `hate` membuat sentimennya berlawanan. Pada percobaan ini, kedekatan vektor belum cukup untuk membedakan sentimen tersebut.

## 4. Tabel Perbandingan

Hasil ketiga pasangan kalimat ditampilkan dalam tabel berikut.

In [ ]:
from IPython.display import display, HTML
from html import escape

baris_tabel = ""
for hasil in hasil_perbandingan:
    baris_tabel += (
        "<tr>"
        f"<td>{escape(hasil['Kalimat 1'])}</td>"
        f"<td>{escape(hasil['Kalimat 2'])}</td>"
        f"<td>{hasil['Cosine Similarity']:.4f}</td>"
        f"<td>{escape(hasil['Interpretasi'])}</td>"
        "</tr>"
    )

display(HTML(
    '<table border="1" cellpadding="8" style="border-collapse:collapse">'
    '<thead><tr><th>Kalimat 1</th><th>Kalimat 2</th>'
    '<th>Cosine Similarity</th><th>Interpretasi</th></tr></thead>'
    f'<tbody>{baris_tabel}</tbody></table>'
))

Kalimat 1,Kalimat 2,Cosine Similarity,Interpretasi
I love AI.,AI is great.,0.7157,Kedua kalimat menyatakan pandangan positif tentang AI.
The sky is blue.,I love pizza.,0.6160,Kedua kalimat membahas topik yang berbeda.
I love this movie.,I hate this movie.,0.9002,"Topiknya sama, tetapi sentimennya berlawanan."


## 5. Ringkasan Hasil

Tabel menunjukkan urutan skor dari yang tertinggi: pasangan tentang film **0,9002**, pasangan tentang AI **0,7157**, lalu pasangan langit dan pizza **0,6160**.

BERT dengan rata-rata embedding token memberi skor berbeda untuk setiap pasangan. Namun, skor tinggi tidak selalu berarti dua kalimat menyampaikan maksud yang sama. Hal ini terlihat pada pasangan tentang film yang memiliki sentimen berlawanan. Karena itu, isi kalimat tetap perlu dibaca saat menafsirkan skor.

In [ ]:
tertinggi = max(hasil_perbandingan, key=lambda x: x["Cosine Similarity"])
terendah = min(hasil_perbandingan, key=lambda x: x["Cosine Similarity"])
print("Skor tertinggi:")
print(f"{tertinggi['Kalimat 1']} / {tertinggi['Kalimat 2']}")
print(f"Cosine similarity: {tertinggi['Cosine Similarity']:.4f}")
print("\nSkor terendah:")
print(f"{terendah['Kalimat 1']} / {terendah['Kalimat 2']}")
print(f"Cosine similarity: {terendah['Cosine Similarity']:.4f}")
print("\nKalimat dengan topik sama dapat memiliki vektor yang dekat meskipun sentimennya berbeda.")
print("Penilaian kemiripan perlu mempertimbangkan skor dan makna pasangan kalimat.")

Skor tertinggi:
I love this movie. / I hate this movie.
Cosine similarity: 0.9002

Skor terendah:
The sky is blue. / I love pizza.
Cosine similarity: 0.6160

Kalimat dengan topik sama dapat memiliki vektor yang dekat meskipun sentimennya berbeda.
Penilaian kemiripan perlu mempertimbangkan skor dan makna pasangan kalimat.


# BAGIAN III ANALISIS TAMBAHAN

## 1. Waktu Pemrosesan BERT, RNN, dan LSTM

Waktu diukur pada CPU untuk input sepanjang 128 dan 256 token. Setiap proses diulang lima kali, lalu diambil nilai tengahnya. Tokenisasi dan pembuatan embedding awal tidak masuk pengukuran.

BERT memakai encoder bawaannya, sedangkan RNN dan LSTM memakai satu layer dengan bobot acak. Karena susunannya berbeda, hasil ini berlaku untuk konfigurasi yang diuji dan tidak menunjukkan perbandingan kualitas model.

In [ ]:
import time
import statistics
import torch.nn as nn

rnn = nn.RNN(model.config.hidden_size, model.config.hidden_size,
             num_layers=1, batch_first=True).eval()
lstm = nn.LSTM(model.config.hidden_size, model.config.hidden_size,
               num_layers=1, batch_first=True).eval()
hasil_waktu = []

with torch.no_grad():
    for panjang in [128, 256]:
        teks_panjang = "Deep learning helps computers understand language. " * panjang
        batch = tokenizer(teks_panjang, max_length=panjang, truncation=True,
                          return_tensors="pt")
        embedding_input = model.embeddings(input_ids=batch["input_ids"])
        # Satu kalimat tanpa padding, jadi semua token ikut diproses.
        proses = {
            "BERT encoder": lambda: model.encoder(embedding_input),
            "RNN 1 layer": lambda: rnn(embedding_input),
            "LSTM 1 layer": lambda: lstm(embedding_input),
        }
        for nama, fungsi in proses.items():
            fungsi()
            durasi = []
            for _ in range(5):
                mulai = time.perf_counter()
                fungsi()
                durasi.append(time.perf_counter() - mulai)
            hasil_waktu.append((nama, batch["input_ids"].shape[1],
                                statistics.median(durasi) * 1000))

print(f"{'Model':<18} {'Token':>8} {'Median waktu (ms)':>20}")
for nama, panjang, waktu in hasil_waktu:
    print(f"{nama:<18} {panjang:>8} {waktu:>20.3f}")

Model                 Token    Median waktu (ms)
BERT encoder            128              420.392
RNN 1 layer             128               17.223
LSTM 1 layer            128               69.418
BERT encoder            256              746.941
RNN 1 layer             256               47.111
LSTM 1 layer            256              166.222


### Hasil Pengukuran Waktu

Pada input 128 token, waktu BERT adalah **420,392 ms**, RNN **17,223 ms**, dan LSTM **69,418 ms**. Saat input bertambah menjadi 256 token, waktunya naik menjadi **746,941 ms**, **47,111 ms**, dan **166,222 ms**. Ketiganya membutuhkan waktu lebih lama saat jumlah token bertambah.

Dalam pengukuran ini RNN paling cepat, disusul LSTM, lalu BERT. Perbandingannya memakai CPU, dan BERT memiliki lebih banyak lapisan daripada RNN serta LSTM satu layer yang diuji. Jadi, hasil ini belum dapat dipakai untuk menyatakan bahwa Transformer selalu lebih lambat.

## 2. Pertanyaan Reflektif

**1. Apa keunggulan self-attention dibanding recurrent connection pada RNN?**

Self-attention dapat melihat hubungan antarkata secara langsung, termasuk kata yang letaknya berjauhan. Pada RNN, informasi diteruskan melalui langkah waktu satu per satu.

**2. Mengapa Transformer dapat diproses secara paralel?**

Dalam satu lapisan encoder, token dapat dihitung bersamaan melalui operasi matriks. Perhitungannya tidak perlu menunggu hidden state token sebelumnya. Namun, hasil satu lapisan tetap diperlukan oleh lapisan berikutnya.

**3. Bagaimana pengaruh ukuran dimensi embedding terhadap performa model?**

Dimensi yang lebih besar dapat menampung lebih banyak fitur, tetapi membutuhkan memori dan komputasi lebih besar. Hasilnya juga dipengaruhi data dan proses pelatihan, sehingga menambah dimensi tidak selalu meningkatkan performa.

### Sumber

- Modul Praktikum Deep Learning 6: Transformer dan Attention Mechanism untuk Pemrosesan Bahasa.
- Dokumentasi BERT: https://huggingface.co/docs/transformers/model_doc/bert
- Dokumentasi cosine similarity: https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cosine_similarity.html